<a href="https://colab.research.google.com/github/gauravtupe-svg/crm-leads-silver-layer/blob/main/GitHub_crm_leads_mview_silver_layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datetime import time
import pandas as pd
import numpy as np
from datetime import datetime , timedelta , date
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
import json
import gspread

from google.oauth2 import service_account
from gspread_dataframe import get_as_dataframe, set_with_dataframe
from gspread_formatting import *

service_account_info = json.loads(
    os.environ["GOOGLE_SERVICE_ACCOUNT_JSON"]
)

creds = service_account.Credentials.from_service_account_info(
    service_account_info,
    scopes=[
        "https://www.googleapis.com/auth/spreadsheets"
    ]
)

gc = gspread.authorize(creds)

Mounted at /content/drive


In [ ]:
import gspread
from gspread_dataframe import get_as_dataframe, set_with_dataframe
from google.oauth2.service_account import Credentials
from google.oauth2 import service_account
from google.cloud import storage
from gspread_formatting import *
creds = service_account.Credentials.from_service_account_file("/content/drive/My Drive/python crediential/python_gsheet_json_2.json",
                                                              scopes=['https://www.googleapis.com/auth/spreadsheets'])
gc = gspread.authorize(creds)

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows',None)

In [ ]:
def gs_to_df(key: str, worksheet_name: str, colm_range: str):
    sheet = gc.open_by_key(key).worksheet(worksheet_name)

    # Read the specified range and pad missing cells
    data = sheet.get(colm_range, pad_values=True)

    df = pd.DataFrame(data[1:], columns=data[0])

    # Replace empty strings with NaN (optional)
    df.replace("", pd.NA, inplace=True)

    return df

In [ ]:
def update_gs_batch_clear(key:str, sheetName:str, df, col_range:list):
    gs = gc.open_by_key(key)
    try:
            worksheet1 = gs.worksheet(sheetName)

    except gspread.exceptions.WorksheetNotFound:
            worksheet1 = gs.add_worksheet(sheetName, rows=100, cols=20)

    worksheet1.batch_clear(col_range)
    set_with_dataframe(worksheet=worksheet1, dataframe=df, include_index=False,
    include_column_header=True, resize=False)

## **Importing Bronz data**

In [ ]:
try :
  # Leads - Speedioo : pulling all leads data from this sheet
  leads_df1 = gs_to_df("1wgmHCqr7e3Gvg9ehUs6Kokt1sdCgNXKUFrtPpKJjfpo","CRM Leads", "A:AE") # all CRM lead from column A:AA
  print("All data imported")
except Exception as a :
  print(a)

All data imported


In [ ]:
leads_df1.head(1)

,ID,Name,Owner,PipelineStage,Phone Number,Created At,City,Source,MMV,Make,Model,Varient,REG Year,MFG Year,Owner Count,HP Status,ODO Meter,Scheduling Team Call Status,ST Connected,ST Cold/Drop,Transfer Date,ST Not Connected,Vehicle Category,Followup 1 Date,Customer Expected Amount,Bike Location,ST Remark,UTM Campaign ID,Registration Number,350 CC,Updated At
0,50571414,Omkar Pawar,<NA>,New Lead,+91 9769269111,23-06-2026 23:31:21,Mumbai,WDA,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [ ]:
leads_df1['ID'].isnull().sum()

np.int64(0)

In [ ]:
leads_df1['Assign_date_time'] = pd.to_datetime(leads_df1['Created At'], format = 'mixed', errors = 'coerce')
leads_df1["Assign_date_time"] = leads_df1["Assign_date_time"].dt.strftime("%H:%M:%S")
leads_df1["Assign_date_time"] = pd.to_datetime(leads_df1["Assign_date_time"], format='%H:%M:%S').dt.time

In [ ]:
# extracting transfer time from transfer date column
# leads_df1['Transfer Time'] = leads_df1['Transfer Date'].astype(str).str[11:]

# converting extracted time data into time format
leads_df1['Transfer Time'] = pd.to_datetime(  leads_df1['Transfer Date'],
                                              errors='coerce',
                                              format='%d-%m-%Y %H:%M:%S'
                                            )

In [ ]:
# leads_df1[leads_df1['ID']== '50693646']

In [ ]:
# columns Renaming
leads_df1 = leads_df1.rename(columns = {'ID': 'crm_led_id', 'Owner': 'lead_owner'})

# filtering the Test Data
filtered_df = leads_df1[
    (~leads_df1['Name'].str.lower().eq('test')) &
    (~leads_df1['Varient'].str.lower().eq('test'))
]

In [ ]:
# filtered_df.columns.tolist()

In [ ]:
# data cleaning
filtered_df['lead_owner'] = filtered_df['lead_owner'].astype(str).str.capitalize()
filtered_df['Name'] = filtered_df['Name'].astype(str).str.capitalize()
filtered_df['City'] = filtered_df['City'].astype(str).str.capitalize()
filtered_df['MMV'] = filtered_df['MMV'].astype(str).str.strip().str.upper()
filtered_df['Make'] = filtered_df['Make'].astype(str).str.strip().str.upper()
filtered_df['Model'] = filtered_df['Model'].astype(str).str.strip().str.upper()
filtered_df['Varient'] = filtered_df['Varient'].astype(str).str.strip().str.upper()

In [ ]:
# filtered_df.columns.tolist()

In [ ]:

# columns data striping
filtered_df['Phone Number'] = filtered_df['Phone Number'].str.strip()

# correcting phone number format
filtered_df['Phone Number'] = (
    filtered_df['Phone Number']
    .fillna('')
    .astype(str)
    .str.strip()
    .str.replace(r'\D', '', regex=True)
    .str[-10:]
)
# filtered_df['Phone Number'] = filtered_df['Phone Number'].astype(str).str[-10:]
# filtered_df['Phone Number'] = filtered_df['Phone Number'].astype('int64')

In [ ]:
# filtered_df[filtered_df['Phone Number'] == '0']

In [ ]:
filtered_df['Phone Number'] = pd.to_numeric( filtered_df['Phone Number'], errors='coerce').astype('Int64')

In [ ]:
# City Mapping

import pandas as pd

# City Mapping

filtered_df['City'] = filtered_df['City'].fillna('').astype(str).str.strip()

# Pune
filtered_df.loc[
    filtered_df['City'].str.lower() == 'pimpri-chinchwad',
    'City'
] = 'Pune'

# Mumbai
filtered_df.loc[
    filtered_df['City'].str.lower().isin(['navi mumbai', 'thane', 'Mumbra', 'mumbra', 'new mumbai']),
    'City'
] = 'Mumbai'

# Bangalore
filtered_df.loc[
    filtered_df['City'].str.lower().isin([
        'bengaluru', 'bangalore', 'bnaglore',
        'bnagalore', 'banagalore', 'banglore'
    ]),
    'City'
] = 'Bangalore'

# Delhi - considers all words containing Delhi
filtered_df.loc[
    filtered_df['City'].str.contains('delhi', case=False, na=False),
    'City'
] = 'Delhi'

# near by delhi area converting into delhi city
filtered_df.loc[
    filtered_df['City'].astype(str).isin([
        'Faridabad', 'Gaziabad', 'Ghaziabad', 'Gurgaon', 'Gurgoan', 'Gurugram', 'Shahdara'
    ]),
    'City'
] = 'Delhi'

In [ ]:
filtered_df['City'].value_counts()

,count
City,
Mumbai,29441
Pune,26287
Bangalore,24002
Delhi,5043
<na>,1499
Pune store,370
<>,348
Up,10
Hyderabad,10


In [ ]:
# Source Mapping
filtered_df['Source'] = filtered_df['Source'].mask(
    filtered_df['Source'].isin(['WDA', 'FB Form', 'HMMV', 'New FB Form']),
    'Meta')

filtered_df['Source'] = filtered_df['Source'].mask(
    filtered_df['Source'].isin([
        'Website - Sell form',
        'Website - Contact form',
        'Website - Test ride',
        'Website - Book now']),
    'Website'
)

In [ ]:
from datetime import datetime
import pandas as pd
import re

def parse_date(x):

    if pd.isna(x) or str(x).strip() == "":
        return None

    x = str(x).strip()

    # Normalize strings like "Jul 12,2026 at 6:55 pm"
    x = re.sub(r',(\d{4})', r', \1', x)   # Add space after comma
    x = re.sub(r'\s+at\s+', ' ', x, flags=re.IGNORECASE)

    formats = [
        # MM/DD/YYYY formats
        "%m/%d/%Y",
        "%m/%d/%Y %H:%M:%S",
        "%m/%d/%Y %I:%M:%S %p",

        # DD-MM-YYYY formats
        "%d-%m-%Y",
        "%d-%m-%Y %H:%M:%S",

        # ISO formats
        "%Y-%m-%d",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%dT%H:%M:%S%z",

        "%d.%m.%Y",
        "%Y/%m/%d",
        "%d %b %Y",
        "%d %B %Y",

        # New formats
        "%b %d, %Y %I:%M %p",      # Jul 12, 2026 6:55 PM
        "%B %d, %Y %I:%M %p",      # July 12, 2026 6:55 PM
    ]

    for fmt in formats:
        try:
            return datetime.strptime(x, fmt).strftime("%Y-%m-%d")
        except:
            pass

    try:
        dt = pd.to_datetime(x, errors='coerce')
        if pd.notna(dt):
            return dt.strftime('%Y-%m-%d')
    except:
        pass

    return None

In [ ]:
filtered_df['Transfer Date'] = filtered_df['Transfer Date'].apply(parse_date)
filtered_df['Lead date'] = filtered_df['Created At'].apply(parse_date)
filtered_df['Followup 1 Date'] = filtered_df['Followup 1 Date'].apply(parse_date)
filtered_df['Updated At'] = filtered_df['Updated At'].apply(parse_date)

In [ ]:
# filtered_df.head()

In [ ]:
filtered_df['Lead date'] = pd.to_datetime(
    filtered_df['Lead date'],
    errors = 'coerce'
)

In [ ]:
filtered_df['New lead Assigned date'] = np.where(
    (
        filtered_df['Assign_date_time'] >= time(19, 0, 0) ) &
         ( filtered_df['Assign_date_time'] <= time(23, 59, 0)
    ) ,

    filtered_df['Lead date'] + pd.Timedelta(days=1),# if date is 07-08-2026 then it would be 08-08-2026
    filtered_df['Lead date']
)

In [ ]:
# filtered_df.tail()

In [ ]:
filtered_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 87266 entries, 0 to 87280
Data columns (total 35 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   crm_led_id                   87266 non-null  object        
 1   Name                         87266 non-null  object        
 2   lead_owner                   87266 non-null  object        
 3   PipelineStage                87137 non-null  object        
 4   Phone Number                 87197 non-null  Int64         
 5   Created At                   87258 non-null  object        
 6   City                         87266 non-null  object        
 7   Source                       87115 non-null  object        
 8   MMV                          87266 non-null  object        
 9   Make                         87266 non-null  object        
 10  Model                        87266 non-null  object        
 11  Varient                      87266 non-null  o

In [ ]:
# filtered_df.head()

In [ ]:
filtered_df = filtered_df.rename(columns = {
    'Updated At' : 'lead_Updated_At'
})

In [ ]:
# columns seleciton

filtered_df = filtered_df[['crm_led_id', 'Name', 'lead_owner', 'PipelineStage', 'Phone Number', 'Lead date', 'City', 'Source', 'MMV', 'Make', 'Model', 'Varient', 'REG Year',
                           'MFG Year', 'Owner Count', 'HP Status', 'ODO Meter', 'Scheduling Team Call Status', 'ST Connected', 'ST Not Connected', 'Transfer Date',
                           'ST Cold/Drop', 'Vehicle Category', 'Followup 1 Date', 'Customer Expected Amount', 'Bike Location', 'ST Remark', 'Created At', 'UTM Campaign ID', '350 CC',
                           'lead_Updated_At', 'New lead Assigned date', 'Transfer Time' ]]

In [ ]:
filtered_df['crm_led_id'] = filtered_df['crm_led_id'].astype('int64')

In [ ]:
filtered_df['is_duplicate'] = (filtered_df.groupby(['crm_led_id', 'Phone Number']).cumcount()+1)

In [ ]:
# filtered_df[filtered_df['crm_led_id'] == 51157100]

In [ ]:
# filtered_df.head()

## **Silver Leads data Exporting**

In [ ]:
# updating data into sheet
try :
  update_gs_batch_clear("11sWA5GHG6aELKiWyIRior1LOINuoVy8jbh2jlRWP3TY", "CRM Leads", filtered_df, ["A:AH"] )
  print("Data Exported!")
except Exception as b :
  print(b)

Data Exported!


## **Conversion Bronz Data Importing**

In [ ]:
try :
  # Conversion - Speedioo : pulling all conversion data from this sheet
  conversion_df1 = gs_to_df("1pEWn8ZMA7ajZVdp1gneqGT55x_csNFVTs9GzQXoq4pY","Conversion", "A:AM") # all conversion from column A:AH
  print("Conversion data imported")
except Exception as a :
  print(a)

Conversion data imported


In [ ]:
# removing the space from text start & end whitespaces
conversion_df1.columns.str.strip()

Index(['created at', 'Customer Name', 'City', 'Make', 'Model', 'Deal ID',
       'Varient', 'Source', 'Phone Number', 'Manf. Year', 'Odo Meter',
       'Pipeline', 'Reg. Year', 'Remarks', 'HP Status', 'Closed Date',
       'Followup', 'Pickup date', 'Actual Value', 'CT Connected',
       'CT Interested', 'Bike Location', 'Offered Prise', 'Pickup Status',
       'CT Call Status', 'Pipeline Statge', 'CT Not Conected',
       'CT Not Interested', 'Followup 1', 'Followup 2', 'Vehicle Category',
       'Bike Reg. Number', 'Customer Expected Amount', 'Owner Id',
       'Customer 1st name', 'Customer last name', 'Agent First name',
       'Agent Last name', 'Updated At'],
      dtype='object')

In [ ]:
# data Cleaning

conversion_df1['Actual Value'] = conversion_df1['Actual Value'].astype(str).str[4:]  # Actual Bike Prise
conversion_df1['Offered Prise'] = conversion_df1['Offered Prise'].astype(str).str[4:]  # Offered Prise

conversion_df1['Bike Reg. Number'] = conversion_df1['Bike Reg. Number'].astype(str).str.strip()
conversion_df1['Bike Reg. Number'] = conversion_df1['Bike Reg. Number'].str.upper()
conversion_df1['Customer Name'] = conversion_df1['Customer Name'].str.capitalize()
conversion_df1['City'] = conversion_df1['City'].str.capitalize()
conversion_df1['Make'] = conversion_df1['Make'].str.upper()
conversion_df1['Model'] = conversion_df1['Model'].str.upper()
conversion_df1['Varient'] = conversion_df1['Varient'].str.upper()

conversion_df1['Manf. Year'] = conversion_df1['Manf. Year'].astype(str).str[:4]

In [ ]:
# date formatting
conversion_df1['created at'] = conversion_df1['created at'].apply(parse_date)
conversion_df1['Closed Date'] = conversion_df1['Closed Date'].apply(parse_date)
conversion_df1['Followup 1'] = conversion_df1['Followup 1'].apply(parse_date)
conversion_df1['Followup 2'] = conversion_df1['Followup 2'].apply(parse_date)
conversion_df1['Closed Date'] = conversion_df1['Closed Date'].apply(parse_date)
conversion_df1['Pickup date'] = conversion_df1['Pickup date'].apply(parse_date)
conversion_df1['Updated At'] = conversion_df1['Updated At'].apply(parse_date)


In [ ]:
conversion_df1.tail()

,created at,Customer Name,City,Make,Model,Deal ID,Varient,Source,Phone Number,Manf. Year,Odo Meter,Pipeline,Reg. Year,Remarks,HP Status,Closed Date,Followup,Pickup date,Actual Value,CT Connected,CT Interested,Bike Location,Offered Prise,Pickup Status,CT Call Status,Pipeline Statge,CT Not Conected,CT Not Interested,Followup 1,Followup 2,Vehicle Category,Bike Reg. Number,Customer Expected Amount,Owner Id,Customer 1st name,Customer last name,Agent First name,Agent Last name,Updated At
59063,2026-10-29,Harshad,Mumbai,CLASSIC,JAWA 42 FJ,4937008,<NA>,Google,9002539666,2026,1800,Conversion Team,2026,<NA>,<NA>,None,<NA>,None,0.0,<NA>,<NA>,Mumbai,0.0,<NA>,<NA>,Open,<NA>,<NA>,None,None,Low MMV,MH43CT5495,"190,000",74175,Harshad,<NA>,Sakshi,Kathole,2026-10-29
59064,2026-10-29,Akash solunke,Pune,<NA>,BAJAJ AVANGER 220,4937017,<NA>,<NA>,7768931717,<NA>,80000,Conversion Team,2013,<NA>,<NA>,None,<NA>,None,0.0,<NA>,<NA>,<NA>,0.0,<NA>,<NA>,Open,<NA>,<NA>,None,None,Low MMV,MH14DW6975,<NA>,74175,AKASH SOLUNKE,<NA>,Sakshi,Deshmukh,2026-10-29
59065,2026-10-29,Rohan zende,Pune,HERO,SPLENDOR,4937018,SPLENDOR,BIKE4SELL,9226164620,2024,32000,Conversion Team,2024,<NA>,<NA>,None,<NA>,None,0.0,<NA>,<NA>,Pune,0.0,<NA>,<NA>,Open,<NA>,<NA>,None,None,Low MMV,MH14LT9171,80K,74175,Rohan Zende,<NA>,Rumina,Dange,2026-10-29
59066,2026-10-29,Suraj abhijeet rokade,Pune,HERO,SPLENDOR+ BKL STRIPE I3S,4937046,PLUS,Google,9209733706,<NA>,40000,Conversion Team,2024,<NA>,<NA>,None,<NA>,None,0.0,<NA>,<NA>,Pune,0.0,<NA>,<NA>,Open,<NA>,<NA>,None,None,Low MMV,MH12WY7488,60K,74175,Suraj abhijeet rokade,<NA>,Arbaz,Sayyad,2026-10-29
59067,2026-10-29,Avinash,Mumbai,<NA>,HONDA DIO 110CC,4937049,<NA>,Meta,9146339939,<NA>,<NA>,Conversion Team,2014,<NA>,<NA>,None,<NA>,None,0.0,<NA>,<NA>,Mumbai,0.0,<NA>,<NA>,Open,<NA>,<NA>,None,None,Low MMV,MH05CF0694,not mentioned,74175,Avinash,<NA>,Pratik,Kadam,2026-10-29


In [ ]:
# concating agent name
conversion_df1['Transfered_to'] = (conversion_df1['Agent First name'] + " " + conversion_df1['Agent Last name']).str.strip()

In [ ]:
conversion_df1 = conversion_df1[['created at', 	'Customer Name', 	'City', 	'Make', 	'Model', 	'Deal ID', 	'Varient', 	'Source', 	'Phone Number',
                                 'Manf. Year', 	'Odo Meter', 	'Pipeline', 	'Reg. Year', 	'Remarks', 	'HP Status', 	'Closed Date', 	'Followup',
                                 'Pickup date', 	'Actual Value', 	'CT Connected', 	'CT Interested', 	'Bike Location', 	'Offered Prise', 	'Pickup Status',
                                 'CT Call Status', 	'Pipeline Statge', 	'CT Not Conected', 	'CT Not Interested', 	'Followup 1', 	'Followup 2', 	'Vehicle Category',
                                 'Bike Reg. Number', 	'Customer Expected Amount', 	'Owner Id', 'Transfered_to', 'Updated At']]

In [ ]:
conversion_df1 = conversion_df1.rename(columns ={
    'Updated At' : 'conversion_Updated_At'} )

In [ ]:
conversion_df1['City'].value_counts()

,count
City,
Mumbai,10639
Bangalore,9914
Pune,9362
Delhi,1099
Thane,252
Navi mumbai,117
New delhi,111
Gurgaon,109
North west delhi,56


## **Updating Silver Conversion data**

In [ ]:
# updating conversion data into sheet
try :
  update_gs_batch_clear("11sWA5GHG6aELKiWyIRior1LOINuoVy8jbh2jlRWP3TY", "CRM Conversion", conversion_df1, ["A:AH"] )
  print("Conversion Data Exported!")
except Exception as b :
  print(b)

Conversion Data Exported!
